In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
df=pd.read_csv(r"C:\Users\ELCOT\Desktop\Sowmiga\Python Workspace\Milestone Project\it_support_raw_dataset.csv")

In [3]:
df.head()

,Ticket_ID,Created_At,Description,Priority,Category,Assigned_Team,Status,First_Response_At,Resolved_At,Response_SLA_Hours,Resolution_SLA_Hours,Actual_Response_Hours,Actual_Resolution_Hours,SLA_Breached
0,TKT0001,2025-06-12 12:53:00,Monitor display is flickering,MEDIUM,Hardware,Endpoint Support,Open,2025-06-12 17:51:26,NaN,4.0,48,4.97,NaN,Yes
1,TKT0002,2025-01-20 07:44:00,Printer is not working,Low,Hardware,Endpoint Support,Closed,2025-01-20 16:27:49,2025-01-26 03:37:47,8.0,72,8.73,139.90,Yes
2,TKT0003,2025-11-04 17:43:00,Office application crashes,Medium,SOFTWARE,Infrastructure,Resolved,2025-11-04 19:52:21,2025-11-09 05:32:58,4.0,48,2.16,107.83,Yes
3,TKT0004,2025-07-12 14:50:00,Monitor display is flickering,Low,Hardware,Endpoint Support,Open,2025-07-13 09:13:38,NaN,8.0,72,18.39,NaN,Yes
4,TKT0005,2025-09-21 08:01:00,Unable to install software,Low,Software,Applications,Open,2025-09-21 11:35:18,NaN,8.0,72,3.57,NaN,No


In [4]:
df.sample(5)

,Ticket_ID,Created_At,Description,Priority,Category,Assigned_Team,Status,First_Response_At,Resolved_At,Response_SLA_Hours,Resolution_SLA_Hours,Actual_Response_Hours,Actual_Resolution_Hours,SLA_Breached
339,TKT0340,2025-03-14 21:58:00,Account locked out,Medium,Account / Access,Infrastructure,Open,2025-03-15 03:27:46,NaN,4.0,48,5.50,NaN,Yes
262,TKT0263,2025-04-12 02:24:00,Laptop will not turn on,Medium,Hardware,Endpoint Support,In Progress,NaN,NaN,4.0,48,1.41,NaN,No
456,TKT0457,2025-08-24 23:33:00,Application login error,Medium,Software,Applications,Resolved,2025-08-25 09:22:23,2025-08-29 00:41:59,4.0,48,9.82,97.15,Yes
286,TKT0287,2025-06-10 14:01:00,VPN connection drops,Low,Network,Network Operations,Resolved,2025-06-10 20:46:15,2025-06-12 03:53:31,8.0,72,6.75,37.88,No
297,TKT0298,2025-02-03 10:54:00,Computer is running slowly,High,Other,Service Desk,Open,2025-02-03 13:05:50,NaN,1.0,24,2.20,NaN,Yes


Tickets sheet contains 512 rows and 14 columns.

In [5]:
df.shape

(512, 14)

In [6]:
print(df.duplicated().sum())

9


In [7]:
print(df["Ticket_ID"].duplicated().sum())

12


In [8]:
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (503, 14)


In [9]:
print(df["Ticket_ID"].duplicated().sum())

3


In [10]:
# TKT0244: Remove the row with a missing or empty description
df = df[~(
    (df["Ticket_ID"] == "TKT0244") &
    (df["Description"].fillna("").str.strip() == "")
)]

# TKT0263: Remove the row with a missing first-response timestamp
df = df[~(
    (df["Ticket_ID"] == "TKT0263") &
    (df["First_Response_At"].isna())
)]

# TKT0351: Keep only one record when the Ticket_ID is repeated
df = df[~(
    (df["Ticket_ID"] == "TKT0351") &
    df.duplicated(subset=["Ticket_ID"], keep="first")
)]

# Verify the final dataset
print("Final dataset shape:", df.shape)
print("Unique Ticket IDs:", df["Ticket_ID"].nunique())
print("Repeated Ticket IDs:", df["Ticket_ID"].duplicated().sum())

Final dataset shape: (500, 14)
Unique Ticket IDs: 500
Repeated Ticket IDs: 0


In [11]:
print("Dataset shape:", df.shape)
print("Unique Ticket IDs:", df["Ticket_ID"].nunique())
print("Repeated Ticket IDs:", df["Ticket_ID"].duplicated().sum())

Dataset shape: (500, 14)
Unique Ticket IDs: 500
Repeated Ticket IDs: 0


In [12]:
df.isnull().sum()

Ticket_ID                    0
Created_At                   0
Description                  7
Priority                     5
Category                     4
Assigned_Team               10
Status                       4
First_Response_At            5
Resolved_At                233
Response_SLA_Hours           0
Resolution_SLA_Hours         0
Actual_Response_Hours        0
Actual_Resolution_Hours    233
SLA_Breached                 2
dtype: int64

In [13]:
missing = df.isnull().sum()

print(missing[missing > 0])

Description                  7
Priority                     5
Category                     4
Assigned_Team               10
Status                       4
First_Response_At            5
Resolved_At                233
Actual_Resolution_Hours    233
SLA_Breached                 2
dtype: int64


In [14]:
df["Description"] = df["Description"].fillna("")

for col in ["Priority", "Category", "Assigned_Team", "Status"]:
    df[col] = df[col].fillna("Unknown")

df["SLA_Breached"] = df["SLA_Breached"].replace({
    "Y": "Yes",
    "N": "No",
    "FALSE": "No",
    "breach": "Yes"
})

df["SLA_Breached"] = df["SLA_Breached"].fillna("Unknown")

Standardize the labels

In [15]:
for col in ["Priority", "Category", "Assigned_Team", "Status"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


Priority:
Priority
Medium      192
High        130
Low         114
Critical     39
MEDIUM        7
high          6
low           5
Unknown       5
critical      2
Name: count, dtype: int64

Category:
Category
Software            144
Network              98
Hardware             92
Account / Access     91
Other                53
SOFTWARE              6
Unknown               4
hardware              4
network               3
other                 3
Account/access        2
Name: count, dtype: int64

Assigned_Team:
Assigned_Team
Applications          139
Service Desk          130
Endpoint Support      103
Network Operations    100
Infrastructure         18
Unknown                10
Name: count, dtype: int64

Status:
Status
Resolved       147
Open           127
Closed         118
In Progress    104
Unknown          4
Name: count, dtype: int64


In [16]:
for col in ["Priority", "Category"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


Priority:
Priority
Medium      192
High        130
Low         114
Critical     39
MEDIUM        7
high          6
low           5
Unknown       5
critical      2
Name: count, dtype: int64

Category:
Category
Software            144
Network              98
Hardware             92
Account / Access     91
Other                53
SOFTWARE              6
Unknown               4
hardware              4
network               3
other                 3
Account/access        2
Name: count, dtype: int64


In [17]:
df["Priority"] = df["Priority"].str.strip().str.title()

df["Category"] = df["Category"].str.strip().str.title()

df["Category"] = df["Category"].replace({
    "Account/Access": "Account / Access"
})

In [18]:
print(df["Priority"].value_counts())
print(df["Category"].value_counts())

Priority
Medium      199
High        136
Low         119
Critical     41
Unknown       5
Name: count, dtype: int64
Category
Software            150
Network             101
Hardware             96
Account / Access     93
Other                56
Unknown               4
Name: count, dtype: int64


In [19]:
print(df["First_Response_At"].isnull().sum())

5


In [20]:
missing_response = df[df["First_Response_At"].isnull()]

display(missing_response[
    ["Ticket_ID", "Created_At", "Status",
     "First_Response_At", "Actual_Response_Hours"]
])

,Ticket_ID,Created_At,Status,First_Response_At,Actual_Response_Hours
33,TKT0034,2025-04-06 06:14:00,Closed,NaN,0.61
101,TKT0102,2025-04-17 00:00:00,In Progress,NaN,1.55
391,TKT0392,2025-05-01 01:32:00,Closed,NaN,0.22
395,TKT0396,2025-04-10 21:06:00,Open,NaN,3.67
435,TKT0436,2025-08-04 20:33:00,Open,NaN,2.26


preprocessing

In [21]:
df["Description"] = df["Description"].str.lower()

Remove special characters


In [22]:

df["Description"] = df["Description"].apply(
    lambda x: re.sub(r"[^a-zA-Z0-9\s]", "", x)
)

Remove extra spaces 


In [23]:
df["Description"] = df["Description"].str.strip().str.replace(r"\s+", " ", regex=True)

In [24]:
display(df[["Description"]].head())

,Description
0,monitor display is flickering
1,printer is not working
2,office application crashes
3,monitor display is flickering
4,unable to install software


In [25]:
df.to_csv("it_support_preprocessed.csv", index=False)

print("Preprocessed dataset saved.")
print("Final dataset shape:", df.shape)

Preprocessed dataset saved.
Final dataset shape: (500, 14)


Text Vectorization (TF-IDF)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)

X_tfidf = tfidf.fit_transform(df["Description"])

print(X_tfidf.shape)

(500, 59)
